# Step 07. What would actually cross an institutional boundary

Cohorts A, B and C stand in for three institutions. This step is the point of the whole exercise:
doing the analysis above without any site sending patient-level data anywhere.

## What weighted gene co-expression network analysis (WGCNA) needs

Exactly one thing: the protein × protein correlation matrix. Every entry of it is a sum over
patients, so it is reconstructed exactly from four matrices that are also sums over patients:

| | |
|---|---|
| `N` | pairwise-complete counts |
| `S` | pairwise column sums |
| `Q` | pairwise sums of squares |
| `G = XᵀX` | the Gram matrix |

Summing these across sites and recombining gives the pooled correlation matrix to machine
precision, not an approximation, not a meta-analytic average. Same tree, same modules, as if the
raw matrices had been pooled.

In [1]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
suff <- function(M) { M <- as.matrix(M)
  list(N = crossprod(!is.na(M)), S = crossprod(!is.na(M), replace(M, is.na(M), 0)),
       Q = crossprod(!is.na(M), replace(M, is.na(M), 0)^2),
       G = crossprod(replace(M, is.na(M), 0))) }

pool <- function(...) { L <- list(...)
  Reduce(function(a, b) Map(`+`, a, b), L) }

pooled_cor <- function(st) {
  n <- st$N; s <- st$S; g <- st$G; q <- st$Q
  cov <- g/n - (s * t(s))/(n * n)
  sdv <- sqrt(q/n - (s/n)^2)
  cov / (sdv * t(sdv))
}

## The round trip

| round | who | what moves |
|---|---|---|
| 1 | each site | the four p × p matrices. No patient appears in them. |
| 2 | aggregator | pools by addition, runs WGCNA once, ships back module definitions, lists of protein names |
| 3 | each site | computes its own eigenproteins and clusters its own patients, locally |

**Patient clustering is never federated, and cannot be.** The sites hold disjoint patients, so there
is no shared vocabulary of objects to cluster, and a patient cluster *is* a set of subject ids.
What federation buys is a common module definition: one module means the same fourteen proteins
at every site, so a claim made at two sites is the same claim rather than a coincidence of naming.
Step 10 performs this, and finds the interferon module bands the same way at all three cohorts
on a pooled definition none of them made alone.

## The constraint

`rank(G) = min(n, p)`.

With 7,288 proteins against roughly 90–100 patients per site, p ≫ n. The Gram matrix is
rank-deficient in exactly the direction that matters: it can be inverted back toward the rows. At
n = 1 the Gram matrix simply *is* the patient's profile.

The release rule is p < n, so the protein panel must be reduced before any Gram leaves a site.
Cluster size has never been the guarantee: Homer *et al.* (2008) recovered individual participants
from allele frequencies pooled over roughly a thousand people.

Two ways forward, neither free:

1. Reduce the panel first, by a criterion agreed across sites and computed from per-protein
   summaries only (means and variances are 1-D and far safer than a Gram). With n ≈ 90 that means
   a panel of at most ~80 proteins, which would have been enough for the interferon module, and
   not enough to discover it.
2. Ship module definitions only, discovered at one site. Loses exactness, keeps the privacy
   boundary trivially safe, and turns the other sites into replication rather than discovery.

The analysis in these notebooks used pooled raw data and is therefore a simulation of what
federation would produce, not a federated run. That distinction belongs in any write-up.

In [2]:
# Demonstration that pooling is exact, on a small panel where p < n is satisfiable.
A <- read.csv(coh("R_cohort-A_log2_combat.csv"), row.names = 1, check.names = FALSE)
B <- read.csv(coh("R_cohort-B_log2_combat.csv"), row.names = 1, check.names = FALSE)
p <- intersect(colnames(A), colnames(B))[1:50]

pooled <- pooled_cor(pool(suff(A[, p]), suff(B[, p])))
direct <- cor(rbind(A[, p], B[, p]))
sprintf("max |pooled - direct| = %.2e", max(abs(pooled - direct)))

[1] "max |pooled - direct| = 2.21e-12"

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [3]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:04 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
